In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install -q "transformers==4.52.4" accelerate bitsandbytes

In [ ]:
!pip install -q langchain-classic

In [ ]:
!pip install -q -U "bitsandbytes>=0.46.1"

In [ ]:
from importlib.metadata import version, PackageNotFoundError

try:
    print("bitsandbytes version:", version("bitsandbytes"))
except PackageNotFoundError:
    print("bitsandbytes is not installed")

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    pipeline,
)

MODEL_ID = "mistralai/Mistral-Nemo-Instruct-2407"

compute_dtype = (
    torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True,
    fix_mistral_regex=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

print("Model and text-generation pipeline loaded.")

In [ ]:
prompt = "Explain what a CV is in one short sentence."

result = generator(
    prompt,
    max_new_tokens=60,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
)

print(result[0]["generated_text"])

In [ ]:
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.prompts import PromptTemplate
import re

response_schemas = [
    ResponseSchema(
        name="full_name",
        description="Candidate's full name as a string. Use an empty string if missing.",
    ),
    ResponseSchema(
        name="email",
        description="Candidate's email as a string. Use an empty string if missing.",
    ),
    ResponseSchema(
        name="education",
        description=(
            "A list of objects. Each object must have degree, institution, and year. "
            "Use an empty list if no education is listed."
        ),
    ),
    ResponseSchema(
        name="skills",
        description="A list of skill names as strings. Use an empty list if none are listed.",
    ),
    ResponseSchema(
        name="experience",
        description=(
            "A list of objects. Each object must have role, company, and years. "
            "Use an empty list if no work experience is listed."
        ),
    ),
]

output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()

def extract_json_block(text):
    matches = re.findall(
        r"```json\s*(.*?)\s*```",
        text,
        re.DOTALL | re.IGNORECASE,
    )
    if matches:
        return matches[-1].strip()
    return text.strip()


def parse_cv(text) -> dict:
    """Extract a structured candidate profile from CV text."""

    if not isinstance(text, str) or not text.strip():
        return {"error": "CV text is empty. Please provide text to parse."}

    prompt_template = PromptTemplate(
        template="""
You extract candidate information from CV text.

Use only facts stated in the CV. Do not guess or invent missing information.
Include only paid or formal employment in experience.
Exclude projects, coursework, volunteering, student organizations, and club roles.
If no paid or formal employment is listed, return an empty experience list.
Return all five required fields.
Use an empty string for a missing full_name or email.
Use an empty list for missing education, skills, or experience.
For each education item, include degree, institution, and year.
For each experience item, include role, company, and years.
Return only valid JSON. Do not add an introduction or explanation.

{format_instructions}

CV text:
<CV>
{cv_text}
</CV>
""",
        input_variables=["cv_text", "format_instructions"],
    )

    for attempt in range(2):
        if attempt == 0:
            prompt = prompt_template.format(
                cv_text=text,
                format_instructions=format_instructions,
            )
        else:
            prompt = prompt_template.format(
                cv_text=text,
                format_instructions=(
                    format_instructions
                    + "\nReturn exactly one JSON object. Do not use Markdown fences. "
                    "Do not include any text before or after the JSON."
                ),
            )

        try:
            generated = generator(
                prompt,
                max_new_tokens=700,
                do_sample=False,
                return_full_text=False,
                pad_token_id=tokenizer.eos_token_id,
            )
            answer = generated[0]["generated_text"]
            return output_parser.parse(extract_json_block(answer))

        except Exception as error:
            if attempt == 1:
                return {
                    "error": (
                        "Could not parse the model response after one retry. "
                        f"Details: {type(error).__name__}: {error}"
                    )
                }

    return {"error": "Could not parse the CV."}


sample_cv = (
    "John Smith - john.smith@email.com / "
    "Education: B.Sc. Computer Science, MIT, 2020 / "
    "Skills: Python, Machine Learning, Data Analysis / "
    "Experience: Software Engineer at Google (2020-2023), "
    "Data Scientist at OpenAI (2023-Present)"
)

sample_result = parse_cv(sample_cv)
print(sample_result)

In [ ]:
!pip install -q fastapi uvicorn

In [ ]:
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
from kaggle_secrets import UserSecretsClient
import hmac
import threading
import time
import uvicorn

secrets = UserSecretsClient()
CV_API_TOKEN = secrets.get_secret("CV_API_TOKEN")

app = FastAPI()

class ParseRequest(BaseModel):
    text: str

@app.post("/parse")
def parse_endpoint(
    request: ParseRequest,
    authorization: str | None = Header(default=None),
):
    expected_token = f"Bearer {CV_API_TOKEN}"

    if authorization is None or not hmac.compare_digest(
        authorization, expected_token
    ):
        raise HTTPException(status_code=401, detail="Invalid or missing token")

    return parse_cv(request.text)

server_thread = threading.Thread(
    target=lambda: uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="info",
    ),
    daemon=True,
)
server_thread.start()

time.sleep(3)
print("FastAPI server started on port 8000.")

In [ ]:
import requests

response = requests.get(
    "http://127.0.0.1:8000/docs",
    timeout=10,
)

print("Status:", response.status_code)

In [ ]:
!pip install -q pyngrok

In [ ]:
from kaggle_secrets import UserSecretsClient
from pyngrok import ngrok

secrets = UserSecretsClient()
ngrok_token = secrets.get_secret("NGROK_AUTHTOKEN")

ngrok.set_auth_token(ngrok_token)
tunnel = ngrok.connect(8000, "http")

public_url = tunnel.public_url
print("Public API URL:", public_url)
print("Parse endpoint:", public_url + "/parse")

In [ ]:
import requests
from kaggle_secrets import UserSecretsClient

api_url = f"{public_url}/parse"
cv_api_token = UserSecretsClient().get_secret("CV_API_TOKEN")

sample_cv = (
    "John Smith - john.smith@email.com / "
    "Education: B.Sc. Computer Science, MIT, 2020 / "
    "Skills: Python, Machine Learning, Data Analysis / "
    "Experience: Software Engineer at Google (2020-2023), "
    "Data Scientist at OpenAI (2023-Present)"
)

response = requests.post(
    api_url,
    json={"text": sample_cv},
    headers={"Authorization": f"Bearer {cv_api_token}"},
    timeout=180,
)

print("Status:", response.status_code)
print(response.json())